In [1]:
import os, sys, json, warnings
warnings.filterwarnings("ignore")
RAIZ = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, RAIZ)
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
from src.comun.rutas import DATA_GENERADO, MODELOS
M = json.load(open(os.path.join(MODELOS, "metricas.json"), encoding="utf-8")) if os.path.isfile(os.path.join(MODELOS, "metricas.json")) else {}
for f in os.listdir(MODELOS):
    if f.startswith("metricas_") and f.endswith(".json") and not f.startswith(("metricas_rag", "metricas_agente")):
        M[f[len("metricas_"):-5]] = json.load(open(os.path.join(MODELOS, f), encoding="utf-8"))
def tabla(d, dec=3):
    return pd.DataFrame(d).T.round(dec) if isinstance(d, dict) else pd.DataFrame(d)


# 03. Pronóstico: saldo a 30 días y proyección por meses

Saldo esperado con banda al 80 %, probabilidad de quedar en rojo, día de riesgo y, por mes, ingresos, gastos y neto de los próximos 6 meses.

## Saldo a 30 días frente a sus baselines

In [2]:
P = M['pronostico']; tabla(P['saldo_30d'])

,mae_s,mae_rel_ingreso,mape_acotado,cobertura_80,ancho_rel
ma3_baseline_plan,1725.961,0.721,0.558,0.722,1.683
perfil_estructural,1832.500,0.769,0.601,0.718,1.712
lightgbm_optimizado,883.527,0.426,0.312,0.806,1.298


In [3]:
print('MAE relativo ponderado por la población objetivo:', {k: round(v, 3) for k, v in P['mae_rel_ponderado'].items()})
print('Optuna:', {k: (round(v, 4) if isinstance(v, float) else v) for k, v in P['optuna'].items()})

MAE relativo ponderado por la población objetivo: {'ma3': 0.721, 'lightgbm': 0.427}
Optuna: {'learning_rate': 0.0401, 'num_leaves': 127, 'min_child_samples': 31, 'colsample_bytree': 0.5329, 'subsample': 0.5458, 'reg_lambda': 15.6426, '_mejor_pinball': 0.1514, '_n_trials': 25}


## Anticipación del rojo (solo clientes con saldo positivo en el origen)

In [4]:
pd.DataFrame({k: v for k, v in P['rojo_30d_anticipacion'].items() if isinstance(v, dict)}).T.round(3)

,pr_auc,roc_auc,brier,ece
lightgbm_calibrado,0.594,0.977,0.024,0.005
saldo_bajo_regla,0.581,0.969,0.027,0.022
perfil_estructural,0.266,0.940,0.061,0.088


In [5]:
tabla(P['dia_de_riesgo'])

,n,mae_dias,acierto_pm3
lightgbm,2134.0,5.401,0.426
estructural,2134.0,7.844,0.373
ma_lineal,2134.0,12.474,0.187


## Equidad: error y cobertura por grupo
Una brecha de cobertura mayor a 5 puntos o un error 1,3 veces mayor en un grupo enciende la alerta.

In [6]:
for g, e in P['equidad_por_grupo'].items():
    print(g, '->', {k: (round(v, 3) if isinstance(v, float) else v) for k, v in e['brechas'].items()}); display(tabla(e['tabla'], 3))

segmento -> {'brecha_cobertura': 0.018, 'razon_mae': 1.583, 'peor_grupo_mae': '0', 'peor_grupo_cobertura': '1', 'alerta': True}


,n,mae_rel,cobertura_80,ancho_rel
0,6800.0,0.572,0.820,1.705
1,16500.0,0.472,0.803,1.344
2,26700.0,0.361,0.805,1.166


region -> {'brecha_cobertura': 0.019, 'razon_mae': 1.205, 'peor_grupo_mae': 'centro', 'peor_grupo_cobertura': 'centro', 'alerta': False}


,n,mae_rel,cobertura_80,ancho_rel
centro,6100.0,0.477,0.794,1.402
lima,22400.0,0.430,0.807,1.286
norte,9100.0,0.415,0.805,1.292
oriente,4000.0,0.396,0.812,1.247
sur,8400.0,0.407,0.812,1.287


rango_edad -> {'brecha_cobertura': 0.019, 'razon_mae': 1.24, 'peor_grupo_mae': '25-34', 'peor_grupo_cobertura': '45-54', 'alerta': False}


,n,mae_rel,cobertura_80,ancho_rel
18-24,6300.0,0.360,0.803,1.237
25-34,16000.0,0.446,0.815,1.305
35-44,13100.0,0.437,0.805,1.301
45-54,8900.0,0.439,0.796,1.342
55+,5700.0,0.403,0.805,1.274


... (2 outputs left out)

exposicion_fen -> {'brecha_cobertura': 0.003, 'razon_mae': 1.055, 'peor_grupo_mae': 'posible', 'peor_grupo_cobertura': 'alta', 'alerta': False}


,n,mae_rel,cobertura_80,ancho_rel
alta,7800.0,0.416,0.805,1.264
baja,17000.0,0.425,0.806,1.310
media,22400.0,0.430,0.807,1.286
posible,2800.0,0.439,0.807,1.420


arquetipo -> {'brecha_cobertura': 0.044, 'razon_mae': 2.993, 'peor_grupo_mae': 'mixto', 'peor_grupo_cobertura': 'mixto', 'alerta': True}


,n,mae_rel,cobertura_80,ancho_rel
base_comision,7500.0,0.322,0.823,1.075
formal_quincena,15000.0,0.246,0.820,0.909
independiente,20000.0,0.486,0.800,1.405
mixto,7500.0,0.736,0.779,2.015


## Remuestreo del riesgo de rojo (clase rara)

In [7]:
tabla(P['remuestreo_rojo'])

,pr_auc,roc_auc,brier,ece
sin ajuste,0.603,0.977,0.024,0.006
pesos de clase,0.603,0.977,0.024,0.007
submuestreo 1:2,0.605,0.977,0.024,0.007
SMOTE 1:2,0.588,0.975,0.024,0.006


## Perturbaciones y evento no visto

In [8]:
tabla(P['perturbaciones'])

,mae_s,mae_rel_ingreso,mape_acotado,cobertura_80,ancho_rel
original,883.527,0.426,0.312,0.806,1.298
ruido_10pct_en_historial,892.458,0.429,0.313,0.815,1.306
faltantes_extra_20pct,1023.448,0.474,0.350,0.797,1.300
clientes_con_cambio_de_patron,822.488,0.388,0.389,0.812,1.252
clientes_sin_cambio_de_patron,895.154,0.434,0.298,0.805,1.307


In [9]:
tabla({k: v for k, v in P['evento_no_visto_confinamiento'].items() if isinstance(v, dict)})

,mae_s,mae_rel_ingreso,mape_acotado,cobertura_80,ancho_rel
ma3,2104.530,1.143,0.863,0.618,1.616
lightgbm_sin_variables_de_evento,1141.175,0.744,0.506,0.739,1.371
lightgbm_con_evento_actual,1120.171,0.740,0.500,0.738,1.247
lightgbm_con_escenario_conocido,1129.217,0.742,0.497,0.737,1.237


## ¿Ayuda el texto de los mensajes?

In [10]:
P['texto_en_el_pronostico']['mejora_mae_rel'], P['texto_en_el_pronostico']['rojo_pr_auc_sin'], P['texto_en_el_pronostico']['rojo_pr_auc_con']

({'media': -0.00103023913466606,
  'ic95': [-0.0017074569241359648, -0.0003460391618902551]},
 0.5939896805971716,
 0.5931777571826967)

## Proyección por meses (ingreso, gasto, neto)

In [11]:
PM_ = M['pronostico_mensual']; print(PM_['ventanas'])
for k, v in PM_['por_horizonte'].items():
    print(k); display(tabla(v, 3))

{'entrena': {'filas': 90000, 'origen_desde': '2020-01-31', 'origen_hasta': '2022-06-30'}, 'ajusta': {'filas': 18000, 'origen_desde': '2022-10-31', 'origen_hasta': '2023-03-31'}, 'calibra': {'filas': 9000, 'origen_desde': '2023-04-30', 'origen_hasta': '2023-06-30'}, 'prueba': {'filas': 61500, 'origen_desde': '2024-01-31', 'origen_hasta': '2025-11-30'}}
lightgbm_recalibracion_continua


,ing_mae_rel,ing_wape,ing_cobertura,gas_mae_rel,gas_wape,gas_cobertura,net_mae_rel,net_cobertura
1,0.387,0.307,0.788,0.101,0.123,0.800,0.417,0.797
2,0.392,0.309,0.796,0.102,0.124,0.800,0.423,0.800
3,0.397,0.312,0.798,0.103,0.125,0.798,0.427,0.804
4,0.405,0.313,0.799,0.104,0.126,0.796,0.436,0.802
5,0.413,0.316,0.792,0.104,0.127,0.797,0.442,0.805
6,0.424,0.316,0.790,0.106,0.129,0.790,0.455,0.807


lightgbm_conformal_fijo


,ing_mae_rel,ing_wape,ing_cobertura,gas_mae_rel,gas_wape,gas_cobertura,net_mae_rel,net_cobertura
1,0.387,0.307,0.632,0.101,0.123,0.776,0.417,0.807
2,0.392,0.309,0.659,0.102,0.124,0.801,0.423,0.797
3,0.397,0.312,0.759,0.103,0.125,0.792,0.427,0.801
4,0.405,0.313,0.761,0.104,0.126,0.794,0.436,0.765
5,0.413,0.316,0.788,0.104,0.127,0.794,0.442,0.769
6,0.424,0.316,0.790,0.106,0.129,0.773,0.455,0.846


mismo_mes_anio_previo


,ing_mae_rel,ing_wape,ing_cobertura,gas_mae_rel,gas_wape,gas_cobertura,net_mae_rel,net_cobertura
1,0.540,0.428,0.860,0.136,0.165,0.825,0.597,0.882
2,0.548,0.431,0.834,0.136,0.164,0.822,0.605,0.863
3,0.557,0.435,0.813,0.135,0.164,0.830,0.615,0.852
4,0.567,0.438,0.810,0.136,0.165,0.835,0.625,0.850
5,0.583,0.443,0.792,0.134,0.164,0.825,0.640,0.835
6,0.608,0.446,0.790,0.134,0.165,0.837,0.664,0.834


promedio_3_meses


,ing_mae_rel,ing_wape,ing_cobertura,gas_mae_rel,gas_wape,gas_cobertura,net_mae_rel,net_cobertura
1,0.551,0.513,0.768,0.112,0.137,0.833,0.573,0.830
2,0.562,0.527,0.754,0.111,0.136,0.818,0.584,0.806
3,0.552,0.515,0.744,0.112,0.136,0.820,0.578,0.801
4,0.557,0.511,0.715,0.112,0.137,0.829,0.584,0.790
5,0.542,0.485,0.738,0.112,0.137,0.833,0.569,0.797
6,0.580,0.506,0.752,0.112,0.137,0.831,0.605,0.818


In [12]:
for g in ('tipo', 'seg', 'region', 'edad'):
    print('por', g); display(tabla(PM_[f'por_{g}']['lightgbm_recalibracion_continua'], 3))

por tipo


,n,ing_mae_rel,ing_cobertura,gas_mae_rel,gas_cobertura,net_cobertura
dependiente,18450.0,0.174,0.777,0.083,0.795,0.797
dependiente_con_comision,9225.0,0.290,0.794,0.088,0.797,0.789
independiente,24600.0,0.511,0.799,0.124,0.801,0.808
mixto,9225.0,0.679,0.813,0.103,0.792,0.810


por seg


,n,ing_mae_rel,ing_cobertura,gas_mae_rel,gas_cobertura,net_cobertura
0,8364.0,0.569,0.808,0.203,0.787,0.799
1,20295.0,0.486,0.813,0.083,0.806,0.819
2,32841.0,0.307,0.779,0.090,0.794,0.793


por region


,n,ing_mae_rel,ing_cobertura,gas_mae_rel,gas_cobertura,net_cobertura
centro,7503.0,0.459,0.785,0.107,0.814,0.794
lima,27552.0,0.396,0.788,0.105,0.796,0.797
norte,11193.0,0.413,0.795,0.103,0.787,0.798
oriente,4920.0,0.348,0.822,0.089,0.800,0.822
sur,10332.0,0.390,0.801,0.103,0.797,0.818


por edad


,n,ing_mae_rel,ing_cobertura,gas_mae_rel,gas_cobertura,net_cobertura
18-24,7749.0,0.337,0.788,0.106,0.785,0.795
25-34,19680.0,0.417,0.799,0.098,0.801,0.802
35-44,16113.0,0.411,0.793,0.100,0.804,0.803
45-54,10947.0,0.433,0.790,0.118,0.798,0.798
55+,7011.0,0.364,0.794,0.097,0.784,0.815
